# 04. Anomaly Transformer와 원인 추적 (XAI)

> **Day 01 — 제조 시계열 AI**
> - 다변량 센서 데이터에서 이상 구간을 탐지하고, Attention 가중치로 어떤 센서가 원인인지 역추적한다.
> - **이 노트북이 회수하는 난제**: ① 해석 가능성 — "왜 고장이라고 판단했는가"

---

## 데이터 구성 — 이 노트북은 세 가지 데이터를 목적에 따라 나눠 씁니다

| 구간 | 데이터 | 사용 이유 |
|---|---|---|
| §1 문제 정의 | 수처리장 펌프 (실데이터, 5개월·고장 7건) | 라벨이 얼마나 희소한지, 라벨 정의가 왜 문제가 되는지는 실제 설비 데이터로 확인해야 한다 |
| §3 AutoEncoder의 한계 | 펌프 테스트베드 (실데이터, SKAB) | Over-generalization은 고장 라벨이 충분한 데이터에서만 관찰된다 |
| §2, §4~§9 | 압출기 8채널 (합성) | 이상 3유형과 원인 센서 정답이 있어야 XAI 결과를 채점할 수 있다 |

실데이터에는 "어느 센서가 원인인가"에 대한 정답이 없습니다. 그래서 문제 인식은 실데이터로,
알고리즘 검증은 원인 정답이 있는 합성 데이터로 나누어 진행합니다.

| | |
|---|---|
| 이 노트북의 역할 | 난제 ① 회수 — 이상 탐지에서 끝내지 않고 "왜"에 답한다 |
| 앞에서 이어받는 것 | NB03에서 구현한 Attention·Encoder 구조를 그대로 재사용 |
| 다음으로 넘기는 것 | "정상 데이터가 충분하다"는 전제 — NB05에서 이 전제가 깨지는 경우를 다룬다 |

---

> **실습 안내**
> `"""채워넣기"""` 가 적힌 셀은 코드 대부분이 이미 있고, `"""채워넣기"""`로 표시된 부분만 채우면 됩니다.
> 나머지 셀은 실행 결과를 확인하며 따라오시면 됩니다.
> 막히는 부분은 손을 들어 주세요.

## 목차
1. 이상탐지 문제 정의 — 실제 펌프 설비 5개월 기록
2. 제조 이상의 3유형 — Point · Contextual · Collective
3. AutoEncoder 접근과 그 한계 — Over-generalization
4. Anomaly Transformer의 핵심 관찰 — Prior · Series · Association Discrepancy
5. Min-Max 전략으로 학습하기
6. 이상 점수와 임계값 설정
7. 불균형 평가 — Accuracy를 버리고 PR로
8. Attention 히트맵 XAI — 원인 센서 특정
9. AE vs Anomaly Transformer 비교

In [ ]:
# 공통 준비 — Colab / 로컬 양쪽에서 동작
import os, random, warnings
import numpy as np
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch 미설치 — 다음 셀에서 설치합니다.")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print(f"Colab 환경: {IN_COLAB}")

In [ ]:
# ══ 실습 자료 위치 — 강사가 배포한 주소로 이 한 줄만 맞추면 됩니다 ══════════
DATA_REPO = "https://raw.githubusercontent.com/leejiyoon52/ai-course/main/day1"

# Google Drive로 배포받았다면, 위 줄 대신 아래 두 줄의 주석을 푸십시오.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_REPO = "file:///content/drive/MyDrive/ai-course/day1"
# ═══════════════════════════════════════════════════════════════════════

import os, shutil, urllib.request
os.environ["MFG_DATA_BASE"] = DATA_REPO          # loaders.py가 이 값을 읽습니다

for f in ["mfg_datagen.py", "loaders.py"]:
    if os.path.exists(f):
        continue
    src = f"{DATA_REPO}/modules/{f}"
    try:
        if src.startswith("file://"):
            shutil.copy(src[len("file://"):], f)
        else:
            urllib.request.urlretrieve(src, f)
        print(f"다운로드 완료: {f}")
    except Exception:
        print(f"⚠️ {f} 를 받지 못했습니다 — 왼쪽 파일 탭에 직접 업로드해 주세요")

import mfg_datagen
import loaders
print(f"실습 모듈 준비 완료 — 자료 위치: {DATA_REPO}")

---
## 1. 이상탐지 문제 정의 — 예측과 무엇이 다른가

이상탐지는 정상 상태의 패턴을 학습한 뒤, 그 패턴에서 벗어나는 시점을 찾는 문제입니다.
NB03의 RUL 예측과는 다음 지점에서 근본적으로 다릅니다.

| | RUL 예측 (NB03) | 이상탐지 (NB04) |
|---|---|---|
| 라벨 | 모든 시점에 있음 (고장까지 남은 사이클) | 거의 없음 — 고장은 몇 건뿐 |
| 학습 방식 | 지도학습 | 정상 데이터만으로 학습(One-class) |
| 판단 | 얼마나 남았는가 | 지금 평소와 다른가 |

실제 설비 데이터로 이 전제를 확인합니다. 수처리장 펌프 1대의 5개월 기록입니다.

In [ ]:
# 펌프 설비 실데이터 로드 — 2018년 5개월, 5분 간격
"""채워넣기"""
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 3.5)
plt.rcParams["axes.grid"] = True

pump = loaders.load_pump()
n = len(pump)
n_broken = int((pump["machine_status"] == """채워넣기""").sum())
n_abnormal = int((pump["machine_status"] != "NORMAL").sum())
print(f"기간: {pump['timestamp'].min()} ~ {pump['timestamp'].max()}")
print(f"전체 시점    : {n:,}")
print(f"고장(BROKEN) : {n_broken}건  → {100 * n_broken / n:.4f}%")
print(f"비정상 전체  : {n_abnormal:,}  → {100 * n_abnormal / n:.2f}%")
print("\n5개월 동안 고장 라벨이 7개뿐입니다. 지도학습으로 풀 수 있는 문제가 아닙니다.")

In [ ]:
# 데이터 검진부터 — NB02에서 만든 내용 그대로
"""채워넣기"""
SENSOR_COLS = [c for c in pump.columns if c.startswith("sensor_")]
miss = (pump[SENSOR_COLS].isna().mean() * 100).round(1)
print("센서별 결측 비율(%)")
print(miss.sort_values(ascending=False).to_string())
print("\nsensor_15는 5개월 내내 값이 없습니다 — 설치만 되고 신호가 없는 센서입니다.")
print("이런 컬럼을 모른 채 모델에 넣으면 학습은 되지만 아무 정보도 얻지 못합니다.\n")

USE = [c for c in SENSOR_COLS if """채워넣기"""]        # 결측 5% 미만만 사용
print(f"사용 센서 {len(USE)}개 | 제외 {sorted(set(SENSOR_COLS) - set(USE))}")
pump_use = pump[["timestamp"] + USE + ["machine_status"]].copy()
pump_use[USE] = pump_use[USE].ffill().bfill()                        # 남은 짧은 결측만 보정
print(f"검진 후 데이터: {pump_use.shape}")

In [ ]:
# 고장 7건이 신호에 어떻게 나타나는가
"""채워넣기"""
import numpy as np

brk_idx = np.where("""채워넣기""")[0]
abn = (pump_use["machine_status"].values != "NORMAL")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, c in zip(axes, USE[:2]):
    ax.plot(pump_use[c].values, lw=0.4)
    for b in brk_idx:
        ax.axvline(b, color="red", lw=1.2)
    ax.set_ylabel(c)
axes[0].set_title("Pump sensors over 5 months (red = BROKEN)")
plt.tight_layout(); plt.show()
print(f"고장 시점 인덱스: {list(brk_idx)}")
print("고장 직후 신호가 바닥으로 떨어집니다 — 설비가 멈췄기 때문입니다.")

### 라벨 정의의 함정 — 무엇을 "이상"이라 부를 것인가

`BROKEN`은 고장이 일어난 순간이고, `RECOVERING`은 수리·재가동 중인 사후 구간입니다.
둘을 합쳐 "이상"이라 부르면 라벨이 6.6%로 늘어 다루기 편해 보이지만, 여기에 함정이 있습니다.

> **현장 노트**
> PdM PoC에서 "AUC 0.99 달성" 보고를 받으면 저는 라벨 정의부터 확인합니다.
> 멈춘 설비를 "이상"이라고 맞히는 모델은 AUC가 거의 1이 나옵니다. 당연합니다,
> 센서값이 전부 바닥이니까요. 하지만 그건 이미 현장 작업자가 알고 있는 사실입니다.
> 실질적인 가치는 고장 나기 전에 아는 것이고, 그 문제는 난이도가 완전히 다릅니다.
> 다음 두 셀에서 그 격차를 숫자로 확인합니다.

In [ ]:
# 함정 실증 — 같은 데이터·같은 점수인데 라벨 정의만 바꿔 본다
"""채워넣기"""
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

Z = StandardScaler().fit_transform(pump_use[USE].values)
naive_score = (Z ** 2).sum(axis=1)          # 단순 z제곱합 — 모델이라 부르기도 민망한 점수

y_post = abn.astype(int)                                       # ① 사후: BROKEN + RECOVERING
y_pre = np.zeros(len(pump_use), dtype=int)                     # ② 예지: 고장 3시간 전
for b in brk_idx:
    y_pre["""채워넣기"""] = 1                            # 5분 × 36 = 3시간
keep = (~abn) | (y_pre == 1)                                   # 사후 회복 구간은 평가에서 제외

print(f"① 사후 라벨 ({100 * y_post.mean():.2f}%)   → ROC-AUC {roc_auc_score(y_post, naive_score):.3f}"
      f" | PR-AUC {average_precision_score(y_post, naive_score):.3f}")
print(f"② 예지 라벨 ({100 * y_pre[keep].mean():.2f}%)   → ROC-AUC {roc_auc_score(y_pre[keep], naive_score[keep]):.3f}"
      f" | PR-AUC {average_precision_score(y_pre[keep], naive_score[keep]):.3f}")
print("\n① 딥러닝은커녕 z-score 합만으로 거의 완벽합니다 — 성능이 아니라 문제 설정이 쉬웠던 것입니다.")
print("② 같은 데이터, 같은 점수인데 무작위 수준입니다. 실질적인 가치는 이쪽에 있습니다.")
print("\n같은 데이터·같은 점수 함수인데 두 지표가 이렇게 갈리는 이유는 모델이 아니라 '무엇을 라벨(정답)로 정의했는가'에 있습니다.")
print("①은 설비가 이미 멈춘 뒤의 상태를 구별하는 문제이고, ②는 멈추기 전 3시간의 조짐만으로 앞일을 맞히는 문제입니다.")
print("즉 타겟 시점을 사후로 잡느냐 예지로 잡느냐에 따라 문제의 난이도 자체가 달라지고, 그 난이도가 성능 상한을 정합니다.")

---
## 2. 제조 이상의 3유형

실제 펌프 데이터로 문제의 성격을 봤습니다. 이제 **알고리즘의 원리를 검증**하려면
정답이 필요합니다 — 어떤 시점이 이상인지, 그리고 **어느 센서가 원인인지**까지.
실데이터에는 그 정답(원인 센서 라벨)이 없으므로, 이후 실습은 **압출기 8채널
합성 데이터**로 진행합니다. 원인 라벨이 있어야 XAI가 맞았는지 채점할 수 있습니다.

| 유형 | 비유 | 특징 |
|---|---|---|
| **Point** | 갑작스런 굉음 | 한 시점이 튄다. 눈에 잘 띈다 |
| **Contextual** | 한여름의 히터 가동 | **값 자체는 정상 범위**인데 맥락상 이상하다 |
| **Collective** | 리듬 자체가 어긋난 구간 | 개별 값은 정상, 구간 전체의 패턴이 깨진다 |

In [ ]:
# 압출기 다변량 데이터 생성 — 이상 3종 + 원인 센서 정답 포함
"""채워넣기"""
raw, labels, rc = mfg_datagen.gen_multivar_anomaly(n="""채워넣기""", seed=SEED)
SENS = list(raw.columns)

print(f"shape {raw.shape} | 센서: {SENS}")
print(f"이상 비율 {100 * labels.mean():.2f}%  ({int(labels.sum())} / {len(labels)} 시점)")
from collections import Counter
print("이상 구간 유형:", dict(Counter(k for _, _, k, _ in rc["segments"])))

In [ ]:
# 3유형을 하나씩 눈으로 확인 — 원인 센서만 그린다
picks = {}
for start, L, kind, sensor in rc["segments"]:
    picks.setdefault(kind, (start, L, sensor))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, kind in zip(axes, ["point", "contextual", "collective"]):
    s0, L, sensor = picks[kind]
    lo, hi = max(0, s0 - 80), s0 + L + 80
    ax.plot(range(lo, hi), raw[sensor].values[lo:hi], lw=0.9)
    ax.axvspan(s0, s0 + L, color="red", alpha=0.2)
    ax.set_title(f"{kind}\n({sensor})")
plt.tight_layout(); plt.show()
print("Point는 튀어서 보이지만, Contextual은 값의 높이만 보면 정상 범위 안에 있습니다.")

In [ ]:
# Contextual 이상은 '이웃 센서와의 관계'가 깨진 것 — 두 센서를 겹쳐 본다
s0, L, sensor = picks["contextual"]
other = [c for c in SENS if c != sensor][0]
lo, hi = max(0, s0 - 120), s0 + L + 120

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(range(lo, hi), raw[sensor].values[lo:hi], label=f"{sensor} (root cause)")
ax.plot(range(lo, hi), raw[other].values[lo:hi], label=other, alpha=0.7)
ax.axvspan(s0, s0 + L, color="red", alpha=0.15)
ax.set_title("Contextual anomaly: value in range, relationship broken")
ax.legend(); plt.tight_layout(); plt.show()
print("붉은 구간에서 두 센서가 반대로 움직입니다. 단일 센서 임계값으로는 절대 잡히지 않습니다.")

---
## 3. AutoEncoder 접근과 그 한계

가장 널리 쓰이는 출발점은 **AutoEncoder(AE)** 입니다.

1. 정상 데이터만으로 "압축했다가 복원하는" 법을 배우게 합니다
2. 이상 데이터는 배운 적이 없으니 복원이 서툴 것이다 → **재구성 오차가 크면 이상**

논리는 깔끔합니다. 그런데 실제 설비 데이터로 돌려 보면 예상 밖의 일이 일어납니다.

§1의 펌프는 5개월에 고장이 7건뿐이라 검증에 쓸 이상이 부족했습니다.
그래서 이번에는 **같은 수순환 펌프 계열의 테스트베드 데이터**를 씁니다.
고장을 의도적으로 유도해 기록한 벤치마크라 라벨이 충분합니다.

In [ ]:
# 펌프 테스트베드 데이터 로드 — 정상 구간과 고장 유도 구간
bench_tr, bench_te, bench_y = loaders.load_anomaly()
BSENS = list(bench_tr.columns)
print(f"학습(정상 가동) {bench_tr.shape} | 평가 {bench_te.shape}")
print(f"평가 구간 이상 비율: {100 * bench_y.mean():.2f}%")

bsc = StandardScaler().fit(bench_tr)
btr = bsc.transform(bench_tr).astype(np.float32)
bte = bsc.transform(bench_te).astype(np.float32)

In [ ]:
# AutoEncoder 정의 — 윈도우를 통째로 압축했다 편다
"""채워넣기"""
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

WIN = 60

def to_windows(a, win=WIN, stride=WIN):
    return np.stack([a[i:i + win] for i in range(0, len(a) - win + 1, stride)])

class AutoEncoder(nn.Module):
    def __init__(self, c, win, hidden=256, latent=128):
        super().__init__()
        self.c, self.win = c, win
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(c * win, hidden), nn.ReLU(),
                                 nn.Linear(hidden, latent))
        self.dec = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                 nn.Linear(hidden, c * win))

    def forward(self, x):
        return self.dec("""채워넣기""").view(-1, self.win, self.c)

demo = AutoEncoder(len(BSENS), WIN)
print(f"입력 {WIN}x{len(BSENS)} = {WIN * len(BSENS)}차원 → 잠재 128차원으로 압축")
print(f"파라미터 수: {sum(p.numel() for p in demo.parameters()):,}")

In [ ]:
# AE 학습 — epoch마다 '재구성 손실'과 '탐지 성능'을 같이 기록한다
# 학습 약 60초 소요 (T4 기준)
import time

B_tr = to_windows(btr, stride=WIN // 4)
B_te = to_windows(bte, stride=1)
b_center = bench_y.values[WIN // 2: WIN // 2 + len(B_te)]      # 윈도우 중앙 시점 라벨

torch.manual_seed(SEED)
ae_b = AutoEncoder(len(BSENS), WIN).to(DEVICE)
opt_b = torch.optim.Adam(ae_b.parameters(), lr=1e-3)
dl_b = DataLoader(TensorDataset(torch.tensor(B_tr)), batch_size=64, shuffle=True)
Bte_t = torch.tensor(B_te).to(DEVICE)

hist = []
t0 = time.time()
for ep in range(20):
    ae_b.train(); tot = 0.0
    for (xb,) in dl_b:
        xb = xb.to(DEVICE)
        opt_b.zero_grad(); loss = ((ae_b(xb) - xb) ** 2).mean()
        loss.backward(); opt_b.step(); tot += loss.item()
    ae_b.eval()
    with torch.no_grad():
        err_b = ((ae_b(Bte_t) - Bte_t) ** 2).mean(dim=(1, 2)).cpu().numpy()
    hist.append((ep + 1, tot / len(dl_b), roc_auc_score(b_center, err_b)))
    if ep % 4 == 0 or ep == 19:
        print(f"epoch {ep + 1:2d} | 재구성 손실 {hist[-1][1]:.4f} | 탐지 ROC-AUC {hist[-1][2]:.3f}")
print(f"학습 시간 {time.time() - t0:.0f}초")

In [ ]:
# ★ 두 곡선을 겹쳐 본다 — 손실이 내려가는 동안 탐지력은 어떻게 움직였는가
h = np.array(hist)
fig, ax1 = plt.subplots(figsize=(10, 3.5))
ax1.plot(h[:, 0], h[:, 1], color="tab:blue", marker="o", ms=3)
ax1.set_xlabel("epoch"); ax1.set_ylabel("reconstruction loss", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(h[:, 0], h[:, 2], color="tab:red", marker="s", ms=3)
ax2.set_ylabel("detection ROC-AUC", color="tab:red")
best_ep = int(h[np.argmax(h[:, 2]), 0])
ax1.axvline(best_ep, ls="--", color="gray")
ax1.set_title("Reconstruction loss (blue) vs detection ability (red)")
plt.tight_layout(); plt.show()

peak, final = h[:, 2].max(), h[-1, 2]
print(f"탐지 성능 정점: epoch {best_ep} (ROC-AUC {peak:.3f}) → 최종 epoch {int(h[-1, 0])} ({final:.3f})")
if final < peak - 0.02:
    print("\n▶ 손실은 끝까지 내려갔는데 탐지력은 꺾였습니다. 이것이 Over-generalization입니다.")
    print("  학습을 더 할수록 AE가 이상 파형까지 복원해 버려 오차 차이가 줄어든 것입니다.")
else:
    print("\n▶ 이 실행에서는 탐지력이 끝까지 유지됐습니다. 이상이 정상 분포에서 충분히 멀었기 때문입니다.")
    print("  Over-generalization은 이상이 정상과 가까울 때 나타납니다 — 다음 셀 설명을 참고하십시오.")
print("\n어느 쪽이든 결론은 같습니다: 손실 곡선은 탐지력을 대변하지 못합니다. 반드시 따로 재야 합니다.")

### Over-generalization — 이상까지 복원해 버리는 현상

AutoEncoder는 입력을 복원하도록 학습될 뿐, 정상과 이상을 구분하도록 학습되지 않습니다.
학습이 길어질수록 모델은 정상 패턴뿐 아니라 처음 접하는 파형까지 점차 복원하게 되고,
그 결과 정상과 이상 사이의 재구성 오차 차이가 줄어듭니다.

**심화되는 조건**

| 이상의 성격 | 탐지력 변화 |
|---|---|
| 정상 분포에서 먼 경우 (설비 정지처럼 값이 급격히 이탈) | 학습이 길어져도 탐지력이 비교적 유지됨 |
| 정상 분포와 가까운 경우 (미세한 마모, 맥락적 이상) | 학습이 길어질수록 탐지력이 저하됨 |

따라서 학습을 오래 진행한 뒤 탐지 성능이 하락했다면, 우선적으로 이 현상을 점검할 필요가 있습니다.
대응 방법은 명확합니다 — 재구성 손실이 아니라 탐지 지표를 기준으로 조기 종료 시점을 결정하는 것입니다.

**더 근본적인 한계**

이 노트북이 다루는 주제와 직접 관련된 한계가 남아 있습니다.
재구성 오차는 이상 여부만 나타낼 뿐 그 원인을 설명하지 못하며, 현장에 제공할 수 있는 근거는 점수 하나로 제한됩니다.
이어서 탐지와 설명을 함께 제공하는 구조를 다룹니다.

In [ ]:
# 실습 데이터 전환 — 원인 센서 정답이 있는 압출기 데이터로
"""채워넣기"""
split = 6000
tr_df = raw.iloc[:split]["""채워넣기"""]        # 전반부 정상만 학습
te_df = raw.iloc[split:]
te_y = labels.iloc[split:].values
te_root = rc["point_root"][split:]

scaler = StandardScaler().fit(tr_df)
tr = scaler.transform(tr_df).astype(np.float32)
te = scaler.transform(te_df).astype(np.float32)

def to_windows(a, win=WIN, stride=WIN):
    return np.stack([a[i:i + win] for i in range(0, len(a) - win + 1, stride)])

X_tr = to_windows(tr, stride=WIN // 4)                     # 학습은 촘촘히
X_te = to_windows(te)                                      # 평가는 겹치지 않게
Y_te = to_windows(te_y.astype(np.float32))
R_te = to_windows(np.array([str(r) for r in te_root], dtype=object))
C = len(SENS)
print(f"X_tr {X_tr.shape} | X_te {X_te.shape} | 평가 이상 비율 {Y_te.mean():.4f}")

In [ ]:
# 압출기 데이터로 AE 기준선 확보 — 두 규모로 학습해 둔다
"""채워넣기"""
# 학습 약 40초 소요 (T4 기준)
dl_ae = DataLoader(TensorDataset(torch.tensor(X_tr)), batch_size=32, shuffle=True)
Xte_t = torch.tensor(X_te).to(DEVICE)
y_flat = Y_te.ravel()

def train_ae(hidden, latent, epochs):
    torch.manual_seed(SEED)
    m = AutoEncoder(C, WIN, hidden=hidden, latent=latent).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(epochs):
        m.train()
        for (xb,) in dl_ae:
            xb = xb.to(DEVICE)
            opt.zero_grad(); """채워넣기""".backward(); opt.step()
    m.eval()
    with torch.no_grad():
        err = ((m(Xte_t) - Xte_t) ** 2).mean(-1).cpu().numpy().ravel()
    return m, err

# ① 뒤에 만들 Anomaly Transformer와 파라미터 규모를 맞춘 AE (공정한 비교 상대)
ae_small, ae_err_small = train_ae(24, 12, epochs=8)
# ② 파라미터를 훨씬 크게 준 AE (비용을 더 쓰면 어디까지 가는지)
ae_big, ae_err_big = train_ae(256, 128, epochs=24)

for name, m, e in [("작은 AE", ae_small, ae_err_small), ("큰 AE", ae_big, ae_err_big)]:
    print(f"{name} ({sum(p.numel() for p in m.parameters()):>7,} params) → "
          f"ROC-AUC {roc_auc_score(y_flat, e):.3f} | PR-AUC {average_precision_score(y_flat, e):.3f}")
print(f"무작위 기준 PR-AUC {y_flat.mean():.4f}")
print("\n모델을 키우면 탐지력이 오릅니다. 하지만 어느 쪽도 '왜 이상인지'는 말하지 못합니다.")

---
## 4. Anomaly Transformer의 핵심 관찰 — Prior · Series · Association Discrepancy

AutoEncoder는 학습이 길어질수록 이상까지 복원해 재구성 오차의 변별력이 떨어지는 한계(Over-generalization)와,
재구성 오차만으로는 이상의 원인을 설명하지 못하는 한계를 함께 가지고 있었습니다.
Anomaly Transformer는 이 두 한계에 대한 대안으로 제시된 방법으로, 다음 관찰에서 출발합니다.

> - **정상 구간은 시간적으로 먼 시점까지 참조합니다.** 주기적으로 반복되는 패턴이 존재하기 때문입니다.
> - **이상 구간은 인접한 시점만 참조합니다.** 유사한 과거 패턴이 존재하지 않기 때문입니다.

이 참조 범위의 차이를 두 개의 분포로 만들어 비교합니다.

| 이름 | 정의 | 의미 |
|---|---|---|
| **Prior-Association** $P$ | 학습되는 가우시안 분포 | 인접 시점에 집중한다는 가정을 표현 |
| **Series-Association** $S$ | 실제 학습된 Attention | 모델이 실제로 어디를 참조하는가 |
| **Association Discrepancy** | 두 분포의 거리 | 이 값이 **작으면 이상** |

$$\text{AssDis}(t) = \text{KL}\big(P_t \,\|\, S_t\big) + \text{KL}\big(S_t \,\|\, P_t\big)$$

정상 시점은 $S$가 멀리까지 퍼져 $P$(인접 집중)와 크게 다릅니다 → AssDis 큼.
이상 시점은 $S$도 옆만 보므로 $P$와 닮습니다 → AssDis 작음.

In [ ]:
# Prior-Association 구현 — 거리에 따라 감쇠하는 가우시안
"""채워넣기"""
def make_prior(win, sigma):
    """시점 간 거리 |i-j| 기반 가우시안. 행마다 합이 1이 되도록 정규화한다."""
    dist = (torch.arange(win)[None, :] - torch.arange(win)[:, None]).float()
    P = """채워넣기"""
    return P / P.sum(-1, keepdim=True)

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, s in zip(axes, [1.0, 3.0, 10.0]):
    ax.imshow(make_prior(40, s), cmap="viridis")
    ax.set_title(f"sigma = {s}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Prior-association: how far each point is assumed to look")
plt.tight_layout(); plt.show()
print("sigma가 작으면 '옆만 본다', 크면 '멀리까지 본다'는 가정입니다.")
print("sigma는 고정값이 아니라 시점마다 모델이 학습합니다.")

In [ ]:
# Association Discrepancy — 두 분포의 대칭 KL 거리
"""채워넣기"""
def assoc_discrepancy(P, S, eps=1e-8):
    """P, S: (B, h, L, L) → 시점별 불일치도 (B, L)"""
    kl_ps = (P * ("""채워넣기""")).sum(-1)   # KL(P||S)
    kl_sp = (S * (torch.log(S + eps) - torch.log(P + eps))).sum(-1)   # KL(S||P)
    return (kl_ps + kl_sp).mean(1)                                    # head 평균

# 검증: 옆만 보는 Attention은 prior와 닮아 AssDis가 작아야 한다
P_demo = make_prior(40, 3.0)[None, None]
S_near = make_prior(40, 3.0)[None, None]                 # 이상 시점처럼 인접만
S_far = make_prior(40, 15.0)[None, None]                 # 정상 시점처럼 멀리까지
print(f"이상처럼 옆만 볼 때 AssDis: {assoc_discrepancy(P_demo, S_near).mean():.4f}")
print(f"정상처럼 멀리 볼 때 AssDis: {assoc_discrepancy(P_demo, S_far).mean():.4f}")
print("→ 이상 시점의 AssDis가 더 작습니다. 이 성질이 탐지의 근거가 됩니다.")

In [ ]:
# Anomaly-Attention 조립 — Series와 Prior를 함께 내놓는 층
"""채워넣기"""
class AnomalyAttention(nn.Module):
    def __init__(self, d_model, n_heads, win):
        super().__init__()
        self.h, self.dk, self.win = n_heads, d_model // n_heads, win
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.sigma = nn.Linear(d_model, n_heads)          # 시점마다 sigma를 학습
        self.out = nn.Linear(d_model, d_model)
        dist = (torch.arange(win)[None, :] - torch.arange(win)[:, None]).float()
        self.register_buffer("dist2", dist ** 2)

    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, -1)
        sp = lambda t: t.view(B, L, self.h, self.dk).transpose(1, 2)
        q, k, v = sp(q), sp(k), sp(v)
        S = """채워넣기"""      # series
        sig = torch.sigmoid(self.sigma(x).transpose(1, 2)) * self.win
        P = torch.exp(-self.dist2 / (2 * (sig.unsqueeze(-1) + 1e-3) ** 2))
        P = P / P.sum(-1, keepdim=True)                                       # prior
        z = (S @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(z), P, S

_z, _P, _S = AnomalyAttention(32, 4, WIN)(torch.randn(2, WIN, 32))
print(f"출력 {tuple(_z.shape)} | prior {tuple(_P.shape)} | series {tuple(_S.shape)}")

In [ ]:
# Anomaly Transformer 조립 — NB03의 Encoder 구조를 그대로 재사용
"""채워넣기"""
def positional_encoding(L, d_model):
    pos = torch.arange(L).unsqueeze(1).float()
    i = torch.arange(0, d_model, 2).float()
    ang = """채워넣기"""
    pe = torch.zeros(L, d_model)
    pe[:, 0::2] = torch.sin(ang); pe[:, 1::2] = torch.cos(ang)
    return pe

class AnomalyTransformer(nn.Module):
    def __init__(self, c, win, d_model=32, n_heads=4, n_layers=2, d_ff=64):
        super().__init__()
        self.emb = nn.Linear(c, d_model)
        self.register_buffer("pe", positional_encoding(win, d_model))
        self.attn = nn.ModuleList([AnomalyAttention(d_model, n_heads, win) for _ in range(n_layers)])
        self.ffn = nn.ModuleList([nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                                nn.Linear(d_ff, d_model)) for _ in range(n_layers)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, c)                 # 재구성 출력

    def forward(self, x):
        z = self.emb(x) + self.pe
        Ps, Ss = [], []
        for a, f, n1, n2 in zip(self.attn, self.ffn, self.ln1, self.ln2):
            o, P, S = a(z)
            z = n1(z + o); z = n2(z + f(z))
            Ps.append(P); Ss.append(S)
        return self.head(z), Ps, Ss

model = AnomalyTransformer(C, WIN).to(DEVICE)
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

---
## 5. Min-Max 전략으로 학습하기

두 분포를 그대로 두면 모델이 편한 쪽으로 맞춰버려 차이가 사라집니다.
그래서 서로 반대 방향으로 최적화하는 두 단계를 번갈아 적용합니다.

| 단계 | 목적 | 손실 |
|---|---|---|
| Minimize | Prior가 Series를 따라가게 (P를 현실에 맞춤) | `재구성 − k · AssDis(P, S.detach())` |
| Maximize | Series는 Prior에서 멀어지게 (정상 시점이 더 멀리 보게) | `재구성 + k · AssDis(P.detach(), S)` |

`detach()`가 핵심입니다. 한 쪽을 고정한 채 다른 쪽만 움직여야 이 반대 방향 최적화가 성립합니다.
그 결과 정상 시점의 AssDis는 커지고, 이상 시점은 옆밖에 볼 게 없어 작게 남습니다.
두 집단의 간격이 벌어지는 것이 이 학습의 목표입니다.

In [ ]:
# Min-Max 학습 — 한 배치에 두 번의 역전파
"""채워넣기"""
# 학습 약 60초 소요 (T4 기준)
K = 3.0
dl = DataLoader(TensorDataset(torch.tensor(X_tr)), batch_size=32, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
for ep in range(8):
    model.train(); tot = 0.0
    for (xb,) in dl:
        xb = xb.to(DEVICE)
        rec, Ps, Ss = model(xb)
        recon = ((rec - xb) ** 2).mean()
        ad_min = torch.stack([assoc_discrepancy(P, S.detach()) for P, S in zip(Ps, Ss)]).mean()
        ad_max = torch.stack([assoc_discrepancy(P.detach(), S) for P, S in zip(Ps, Ss)]).mean()
        opt.zero_grad()
        ("""채워넣기""").backward(retain_graph=True)     # minimize 단계
        ("""채워넣기""").backward()                      # maximize 단계
        opt.step(); tot += recon.item()
    print(f"epoch {ep + 1}/8 | 재구성 손실 {tot / len(dl):.4f} | {time.time() - t0:.0f}s")

In [ ]:
# 사전학습 체크포인트 로드 — 없으면 방금 학습한 데모 모델로 진행
"""채워넣기"""
state = loaders.load_checkpoint("""채워넣기""")
if state is not None:
    model.load_state_dict(state)
    print("체크포인트를 반영했습니다 — 이후 히트맵 품질이 안정적입니다.")
model.eval()
print("모델 준비 완료")

---
## 6. 이상 점수와 임계값 설정

두 신호를 결합해 최종 점수를 만듭니다.

$$\text{Score}(t) = \underbrace{\text{Softmax}\big(-\text{AssDis}\big)_t}_{\text{옆만 보는 시점일수록 큼}} \times \underbrace{\lVert x_t - \hat{x}_t \rVert^2}_{\text{재구성 오차}}$$

재구성 오차만으로는 부족하고(AE의 한계), AssDis만으로도 부족합니다.
"닮은 과거가 없으면서 동시에 복원도 안 되는 시점"이 진짜 이상입니다.

> **구현 주의 — 정규화 범위**
> 위 softmax를 윈도우 하나 안에서 계산하면 안 됩니다.
> 이상이 한 건도 없는 깨끗한 윈도우에서도 softmax가 억지로 최댓값을 만들어
> 허위 알람이 됩니다. 두 신호 모두 전체 구간 기준으로 표준화한 뒤 더합니다.
> 논문 구현도 시퀀스 전체에 걸쳐 정규화합니다.

### 그리고 어디서 자를 것인가 — 임계값은 알람 민감도 다이얼입니다

> 알람 민감도 다이얼 — 돌릴수록 오탐과 미검출이 서로 반대로 움직입니다.

순서를 뒤집는 것이 실무의 요령입니다. "성능이 가장 좋은 임계값"을 찾는 것이 아니라,
하루에 감당할 수 있는 알람 건수를 먼저 정하고 거기에 맞춰 자릅니다.
알람을 받는 사람이 정해진 인원이기 때문입니다.

---
## 7. 불균형 평가 — Accuracy를 버리고 PR로

NB01에서 예고한 내용입니다. 이상 비율 1% 남짓인 데이터에서
Accuracy는 성적표 역할을 하지 못합니다. 임계값을 정한 뒤 바로 이어서 확인합니다.

In [ ]:
# 이상 점수 산출 — AssDis와 재구성 오차를 전체 기준으로 표준화해 결합
"""채워넣기"""
def anomaly_score(model, X, batch=64):
    """반환: score (N, L), sensor_err (N, L, C), assdis (N, L), series_attn (N, L, L)"""
    model.eval(); Es, Ad, At = [], [], []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.tensor(X[i:i + batch]).to(DEVICE)
            rec, Ps, Ss = model(xb)
            Es.append(((rec - xb) ** 2).cpu().numpy())                # (B, L, C) 센서별 오차
            ad = torch.stack([assoc_discrepancy(P, S) for P, S in zip(Ps, Ss)]).mean(0)
            Ad.append(ad.cpu().numpy())
            At.append(Ss[-1].mean(1).cpu().numpy())                   # head 평균 Attention
    sensor_err = np.concatenate(Es); assdis = np.concatenate(Ad); attn = np.concatenate(At)
    err = sensor_err.mean(-1)                                          # (N, L)
    z = lambda v: (v - v.mean()) / (v.std() + 1e-9)                    # 전체 구간 기준 표준화
    score = """채워넣기"""                                        # 두 신호를 같은 저울에 올려 합산
    return score, sensor_err, assdis, attn

score, sensor_err, assdis_te, attn = anomaly_score(model, X_te)
print(f"score {score.shape} | 센서별 오차 {sensor_err.shape} | attention {attn.shape}")
print(f"정상 시점 평균 AssDis {assdis_te.ravel()[y_flat == 0].mean():.4f} | "
      f"이상 시점 평균 AssDis {assdis_te.ravel()[y_flat == 1].mean():.4f}")

In [ ]:
# AE vs Anomaly Transformer — 같은 압출기 데이터에서 직접 비교
comp_ad = pd.DataFrame({
    "ROC-AUC": [roc_auc_score(y_flat, ae_err_small), roc_auc_score(y_flat, ae_err_big),
                roc_auc_score(y_flat, score.ravel())],
    "PR-AUC": [average_precision_score(y_flat, ae_err_small), average_precision_score(y_flat, ae_err_big),
               average_precision_score(y_flat, score.ravel())],
}, index=["AutoEncoder (동급 규모)", "AutoEncoder (크게 확대)", "Anomaly Transformer"]).round(3)
print(comp_ad.to_string())
print(f"\n무작위 기준 PR-AUC: {y_flat.mean():.4f}")

at_pr = average_precision_score(y_flat, score.ravel())
best_ae_pr = max(average_precision_score(y_flat, ae_err_small), average_precision_score(y_flat, ae_err_big))
if at_pr > best_ae_pr:
    print("Anomaly Transformer가 두 AE보다 높은 PR-AUC를 보입니다 — 재구성 오차 외의 신호가 보탬이 되었습니다.")
else:
    print("이번 실행에서는 AE가 Anomaly Transformer와 대등하거나 더 높습니다.")
    print("파라미터 규모까지 맞춘 공정한 비교는 §10에서 해석 가능성·비용까지 포함해 다시 확인합니다.")

In [ ]:
# 임계값 설정 — 목표 오탐률(1.2%) 기준 분위수 임계값 (시간 관계상 §6 상세 설명·스윕은 생략)
from sklearn.metrics import precision_score, recall_score, f1_score
TARGET_ALARM_RATE = 0.012
s_flat = score.ravel()
threshold = np.quantile(s_flat, 1 - TARGET_ALARM_RATE)
pred = (s_flat >= threshold).astype(int)

In [ ]:
# Accuracy의 무의미함 실증 + 제대로 된 지표 (§7 불균형 평가)
"""채워넣기"""
from sklearn.metrics import precision_recall_curve, roc_curve, auc

acc_all_normal = (y_flat == 0).mean()
acc_model = """채워넣기"""
print(f'"전부 정상" 예측의 정확도 : {100 * acc_all_normal:.2f}%')
print(f"우리 모델의 정확도       : {100 * acc_model:.2f}%")
print("→ 두 숫자가 거의 같습니다. Accuracy로는 두 모델을 구분조차 못 합니다.\n")

print(f"Precision {precision_score(y_flat, pred, zero_division=0):.3f} | "
      f"Recall {recall_score(y_flat, pred, zero_division=0):.3f} | "
      f"F1 {f1_score(y_flat, pred, zero_division=0):.3f}")
print(f"ROC-AUC {roc_auc_score(y_flat, s_flat):.3f} | "
      f"PR-AUC {average_precision_score(y_flat, s_flat):.3f} "
      f"(무작위 기준 {y_flat.mean():.4f})")

In [ ]:
# ROC vs PR — 불균형에서 어느 곡선을 봐야 하는가
prec, rec, _ = precision_recall_curve(y_flat, s_flat)
fpr, tpr, _ = roc_curve(y_flat, s_flat)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "r--", lw=0.8)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title(f"ROC (AUC {auc(fpr, tpr):.3f}) — looks fine")
axes[1].plot(rec, prec)
axes[1].axhline(y_flat.mean(), color="r", ls="--", lw=0.8)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR (AP {average_precision_score(y_flat, s_flat):.3f}) — honest")
plt.tight_layout(); plt.show()
print("ROC의 FPR 분모는 다수인 정상 시점입니다. 오탐 몇 건은 분모에 묻혀 곡선이 좋아 보입니다.")
print("PR은 '알람 중 진짜 비율'을 직접 보여 줍니다 — 현장이 체감하는 숫자입니다.")

---
## 8. Attention 히트맵 XAI — 원인 센서 특정

> Attention 히트맵을 읽는 일은 알람이 울린 순간, 어느 계기판을 보고 있었는지 되짚는 작업입니다.

여기가 이 노트북의 핵심이자 **난제 ① 해석 가능성의 회수 지점**입니다.
두 단계로 답합니다.

1. **언제·왜 이상인가** — Attention 행렬(시점 × 시점)을 정상 구간과 나란히 비교
2. **어느 센서 때문인가** — 센서별 기여도를 뽑아 Root Cause 특정

In [ ]:
# ① 정상 윈도우 vs 이상 윈도우의 Attention 지도 비교
"""채워넣기"""
w_anom = int("""채워넣기""")              # 이상이 가장 많은 윈도우
w_norm = int("""채워넣기""")              # 완전 정상 윈도우

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, w, name in [(axes[0], w_norm, "NORMAL window"), (axes[1], w_anom, "ANOMALOUS window")]:
    im = ax.imshow(attn[w], cmap="viridis")
    ax.set_title(f"{name} (#{w})")
    ax.set_xlabel("attends to (time)"); ax.set_ylabel("query time")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

diag_n = np.mean([attn[w_norm][i, max(0, i-2):i+3].sum() for i in range(WIN)])
diag_a = np.mean([attn[w_anom][i, max(0, i-2):i+3].sum() for i in range(WIN)])
print(f"인접 ±2시점에 쏠린 비중 — 정상 {diag_n:.3f} / 이상 {diag_a:.3f}")
if diag_a > diag_n:
    print("이상 윈도우일수록 대각선(=바로 옆)에 더 쏠립니다 — 참조할 과거가 없기 때문이라는 가설과 일치합니다.")
else:
    print("\n이번 실행에서는 이상 윈도우의 대각선 집중도가 오히려 정상보다 낮습니다.")
    print("옆 시점 대신 특정 시점 하나에 여러 query가 함께 쏠리는 패턴일 수 있습니다 — 위 히트맵의 세로줄을 확인하십시오.")
    print("두 윈도우 각 1개만 비교한 것이므로, 다음 셀의 전체 분포 비교와 함께 판단하는 것이 안전합니다.")
    print("\n※ 이런 결과가 나오는 것 자체는 정상입니다. Association Discrepancy는 '이상 = 닮은 과거가 없는 패턴'을")
    print("전제로 설계됐는데, 이번 실습 데이터의 이상(특히 값 자체가 튀거나 위상만 뒤집힌 유형)은 재구성 오차만으로도")
    print("이미 잘 잡히는 값 수준 왜곡이라 이 가정이 항상 깨끗하게 들어맞지 않을 수 있습니다.")

In [ ]:
# AssDis가 정상과 이상을 실제로 갈라놓았는지 숫자로 확인 (분포 시각화는 생략)
mean_n = assdis_te.ravel()[y_flat == 0].mean()
mean_a = assdis_te.ravel()[y_flat == 1].mean()
print(f"평균 AssDis — 정상 {mean_n:.3f} / 이상 {mean_a:.3f}")
if mean_a < mean_n:
    print("이상 시점이 더 작은 값에 몰려 있습니다 — Min-Max 학습이 의도대로 작동했습니다.")
else:
    print("\n이번 실행에서는 이상 시점의 평균 AssDis가 오히려 더 큽니다 — 의도한 방향과 반대입니다.")
    print("두 분포가 충분히 갈라지지 않았다는 뜻이며, 앞서 −AssDis 단독 성능이 낮았던 것과 같은 원인")
    print("(짧은 학습 epoch, 폴백 합성 데이터)일 가능성이 있습니다. 실제 데이터·충분한 epoch에서는 다시 확인이 필요합니다.")
    print("\n※ 근본적으로는, 이 실습 데이터의 이상 유형(Point의 순간 스파이크, Contextual의 위상 반전)이")
    print("값 수준에서 이미 두드러져 재구성 오차만으로 대부분 설명되고, Association Discrepancy가 겨냥하는")
    print("'주기적으로 반복되는 패턴이 없다'는 신호는 상대적으로 약하게 나타나는 데이터라는 뜻이기도 합니다.")
    print("논문의 가정이 모든 데이터에 똑같이 강하게 들어맞지는 않는다는 것 자체가 실무적으로 중요한 확인입니다.")

In [ ]:
# ② 센서별 기여도 추출 — 어느 계기판이 문제였는가
"""채워넣기"""
flat_root = R_te.ravel()
flat_sensor_err = sensor_err.reshape(-1, C)

alarms = np.where((s_flat >= threshold) & (y_flat == 1))[0]        # 제대로 잡은 알람
pred_root = np.array(SENS)["""채워넣기"""]

print(f"정탐 알람 {len(alarms)}건에 대해 원인 센서를 지목합니다\n")
for i in alarms[:5]:
    contrib = flat_sensor_err[i] / flat_sensor_err[i].sum()
    order = np.argsort(-contrib)[:3]
    top = " / ".join(f"{SENS[j]} {100 * contrib[j]:.0f}%" for j in order)
    hit = "적중" if pred_root[i] == flat_root[i] else f"오답(정답 {flat_root[i]})"
    print(f"  시점 {i:5d} → {top}   [{hit}]")

# 정답 라벨로 채점 — XAI도 검증 대상입니다
hit1 = (pred_root[alarms] == flat_root[alarms]).mean()
top2 = np.argsort(-flat_sensor_err, axis=1)[:, :2]
hit2 = np.mean([flat_root[i] in np.array(SENS)[top2[i]] for i in alarms])
print(f"\n원인 센서 top-1 적중률 {hit1:.3f} | top-2 {hit2:.3f} | 무작위 기준 {1 / C:.3f}")
print("해석이 그럴듯해 보이는 것과 실제로 맞는 것은 다릅니다. 정답이 있을 때 반드시 채점해야 합니다.")

In [ ]:
# 이상 구간의 센서별 기여도 히트맵 — 가로축 시간, 세로축 센서
w = w_anom
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2]})
axes[0].plot(score[w], lw=1.2, color="tab:red")
axes[0].axhline(threshold, ls="--", color="gray")
axes[0].set_ylabel("score")
axes[0].set_title(f"Window #{w}: anomaly score and per-sensor contribution")
im = axes[1].imshow(sensor_err[w].T, aspect="auto", cmap="magma")
axes[1].set_yticks(range(C), SENS, fontsize=8)
axes[1].set_xlabel("time step")
plt.colorbar(im, ax=axes[1], fraction=0.03)
plt.tight_layout(); plt.show()
print("점수가 솟은 시각의 세로줄에서 가장 밝은 칸 — 그 센서가 Root Cause 후보입니다.")
print("현장 엔지니어에게 건네는 것은 '이상입니다'가 아니라 이 그림이어야 합니다.")

> **현장 노트**
> Attention 히트맵을 현장 엔지니어에게 처음 보여 줬을 때 반응은 "그래서 뭐요?" 였습니다.
> 히트맵 자체는 엔지니어의 언어가 아니기 때문입니다.
> 그런데 같은 결과를 **"3번 배럴 온도가 이상 점수의 62%를 차지합니다"** 라는 한 줄로 바꿔
> 전달하자 대화가 달라졌습니다. "그 존 히터 작년에 갈았는데" 하고 곧바로 가설이 나왔습니다.
> XAI의 가치는 그림의 화려함이 아니라 **엔지니어가 자기 경험과 대조할 수 있는 형태**로
> 번역되는 데 있습니다. 모델 출력의 마지막 한 단계는 언제나 번역입니다.

---
## 9. AE vs Anomaly Transformer

세 축으로 정리합니다 — **탐지 성능 / 해석 가능성 / 계산 비용.**

> **비교의 첫 번째 규칙: 예산을 맞춰라.**
> 파라미터가 십수 배 큰 모델을 세 배 오래 학습시킨 뒤 "이 모델이 낫다"고 말하는 것은
> 비교가 아닙니다. 아래 표에는 **AT와 같은 규모로 맞춘 AE**를 먼저 놓고,
> **훨씬 크게 키운 AE**(AT 대비 약 17배)를 참고로 함께 실었습니다.

### 표를 보기 전에 — 지표를 어떻게 읽는가

| 지표 | 무엇을 재는가 | 무작위일 때 | 이상탐지에서의 의미 |
|---|---|---|---|
| **ROC-AUC** | 이상 시점이 정상 시점보다 높은 점수를 받을 확률 | 항상 **0.5** | 순위를 잘 매기는가. 불균형에 둔감해 실제보다 후하게 나온다 |
| **PR-AUC** | 알람을 울렸을 때 그것이 진짜일 비율(정밀도)의 평균 | **이상 비율**(여기서는 약 0.014) | 현장이 체감하는 값. 기저율이 낮으면 절대값도 낮게 나온다 |
| **원인 적중률** | 지목한 원인 센서가 실제 원인과 일치한 비율 | **1/센서 수** = 0.125 | 알람에 근거가 따라오는가 |

**PR-AUC를 읽을 때 반드시 기저율과 함께 보십시오.**
이상 비율이 1.4%인 데이터에서 무작위 추측의 PR-AUC는 0.014입니다.
따라서 PR-AUC 0.5는 "절반밖에 못 맞혔다"가 아니라 **무작위 대비 약 35배**라는 뜻입니다.
같은 이유로 **서로 다른 데이터셋의 PR-AUC를 직접 비교하면 안 됩니다** — 기저율이 다르기 때문입니다.

그래서 아래 표의 첫 줄에 모델 없이 단순 통계만 쓴 베이스라인을 넣었습니다.
이 베이스라인을 넘지 못하면 모델을 쓸 이유가 없습니다.

In [ ]:
# 최종 비교표 — 베이스라인부터 순서대로
"""채워넣기"""
# 베이스라인: 모델 없이 이동중앙값에서 얼마나 벗어났는가 (채널 최댓값)
med = pd.DataFrame(X_te.reshape(-1, C)).rolling(11, center=True, min_periods=1).median()
baseline = """채워넣기"""
rand_pr = y_flat.mean()
base_pr = average_precision_score(y_flat, baseline)

def row(name, score, n_params, basis, hit):
    pr = average_precision_score(y_flat, score)
    return {"모델": name, "파라미터": n_params,
            "ROC-AUC": round(roc_auc_score(y_flat, score), 3),
            "PR-AUC": round(pr, 3),
            "PR/무작위": f"{pr / rand_pr:.0f}배",
            "PR/베이스라인": f"{pr / base_pr:.1f}배" if base_pr > 0 else "-",
            "해석 근거": basis, "원인 적중률": hit}

comp = pd.DataFrame([
    row("① 단순 통계 베이스라인", baseline, "0", "없음", "-"),
    row("② AutoEncoder (동급 규모)", ae_err_small,
        f"{sum(p.numel() for p in ae_small.parameters()):,}", "재구성 오차뿐", "-"),
    row("③ Anomaly Transformer", s_flat,
        f"{sum(p.numel() for p in model.parameters()):,}",
        "Attention 지도 + 센서별 기여도", f"{hit1:.2f} (무작위 {1 / C:.2f})"),
    row("④ AutoEncoder (크게 확대)", ae_err_big,
        f"{sum(p.numel() for p in ae_big.parameters()):,}", "재구성 오차뿐", "-"),
])
print(f"무작위 기준 PR-AUC: {rand_pr:.4f}  (이상 비율과 같습니다)\n")
comp

**표를 읽는 법 — 네 줄이 하나로 이어집니다**

1. **단순 통계도 만만치 않습니다.** Point 이상(뾰족한 스파이크)은 모델 없이도 잡힙니다.
   여기를 못 넘으면 모델을 도입할 이유가 없습니다.
2. **같은 예산이면 구조가 이깁니다.** 파라미터가 비슷할 때 Anomaly Transformer가
   AE를 크게 앞섭니다. "닮은 과거가 있는가"라는 질문이 재구성 오차보다 강한 신호이기 때문입니다.
   특히 값이 정상 범위인 **Contextual 이상**에서 격차가 벌어집니다 — 단순 통계로는 손도 못 대는 유형입니다.
3. **AE도 크게 키우면 따라옵니다.** 다만 파라미터 17배와 학습 시간을 지불해야 하고,
   §3에서 봤듯 크게 키울수록 Over-generalization 위험도 함께 커집니다.
4. **그렇게 해도 AE는 "왜"를 말하지 못합니다.** 원인 센서 지목은 AT만 해냈고,
   이것이 오늘 회수한 난제 ① 그 자체입니다.

**그래도 AE를 고를 때가 있습니다.** 이상이 명백한 형태로 나타나는 공정
(오늘 앞부분의 펌프 정지처럼)에서는 AE로 충분하고, 가볍고 구현 부담이 적다는 것이
실제 이점이 됩니다. **알고리즘의 우열은 이상의 성격이 정합니다** —
Point·Contextual 이상이 흩어져 있으면 Association 기반이 유리하고,
구간 전체가 다른 상태로 바뀌면 재구성 오차만으로도 충분합니다.

> ※ 학습 데이터가 적고 이상이 희소해 숫자는 실행마다 흔들립니다.
> 한 번의 실행으로 순위를 단정하지 말고, 우리 데이터에서 여러 번 재보고 판단하십시오.

> **현장 노트**
> 제가 본 이상탐지 PoC의 절반은 "어떤 모델을 쓸까"에서 시작해 실패했습니다.
> 순서가 반대입니다. **우리 라인의 불량이 Point인가 Contextual인가 Collective인가**,
> 라벨은 언제 붙는가, 오탐과 미검출 중 무엇이 더 비싼가 — 이 세 질문에 답한 뒤에
> 모델을 고르면 후보는 대개 한두 개로 줄어듭니다.

---
## Self-check

### Q1. AutoEncoder의 Over-generalization이란 무엇이며, 어떤 조건에서 심해집니까?

<details>
<summary>정답 보기</summary>

- AE는 "입력을 잘 복원하라"만 배웠을 뿐 "이상은 복원하지 말라"는 배운 적이 없습니다.
- 학습이 길어질수록 처음 보는 이상 파형까지 복원해, 정상과 이상의 오차 차이가 줄어듭니다.
- 실습의 펌프 테스트베드에서도 재구성 손실은 계속 내려가는데 탐지 ROC-AUC는 정점을 찍고 꺾였습니다.
- 다만 **항상 일어나지는 않습니다.** 이상이 정상 분포에서 멀수록(설비 정지처럼) AE는 잘 버팁니다.
  이상이 정상과 가까울수록 심해지므로, 우리 공정의 이상이 어느 쪽인지가 판단 기준입니다.
- 실무 대응은 하나입니다 — **손실이 아니라 탐지 지표로 조기 종료**를 판단합니다.

</details>

---

### Q2. Association Discrepancy가 이상 시점에서 작아지는 이유를 설명하십시오.

<details>
<summary>정답 보기</summary>

- 정상 시점은 주기적으로 닮은 과거가 있어 Attention(Series)이 멀리까지 퍼집니다. 인접 집중을 가정한 Prior와 크게 달라 AssDis가 커집니다.
- 이상 시점은 과거에 닮은 순간이 없어 바로 옆 시점하고만 대화합니다. 그 결과 Series가 Prior와 닮아 AssDis가 작아집니다.
- Min-Max 학습은 Prior를 현실에 맞추면서(minimize) Series는 Prior에서 밀어내(maximize) 두 집단의 간격을 넓힙니다.

</details>

---

### Q3. 이상 비율 1%인 데이터에서 ROC 곡선보다 PR 곡선을 보라고 하는 이유는 무엇입니까?

<details>
<summary>정답 보기</summary>

- ROC의 FPR은 분모가 다수인 정상 시점이라, 오탐이 수십 건 늘어도 곡선이 거의 움직이지 않습니다.
- PR의 Precision은 "울린 알람 중 진짜 비율"을 직접 보여 주며, 이것이 현장이 체감하는 숫자입니다.
- 무작위 기준선도 다릅니다 — ROC는 항상 0.5지만 PR은 이상 비율(여기서는 약 0.01)이라 비교 기준이 정직합니다.

</details>

---

### Q4. [현장 판단] 모델이 "지금 이상입니다"라고 알렸습니다. 이 출력을 그대로 현장에 전달하면 안 되는 이유는 무엇이며, 무엇을 덧붙여야 합니까?

<details>
<summary>정답 보기</summary>

- 근거 없는 알람은 검증할 방법이 없어 무시되고, 반복되면 알람 피로로 이어져 진짜 알람까지 묻힙니다.
- **어느 센서가 얼마나 기여했는지**(원인 센서와 비중), 언제부터 시작됐는지(탐지 지연), 과거 유사 사례를 함께 전달해야 합니다.
- 히트맵 그대로가 아니라 "3번 배럴 온도가 이상 점수의 62%"처럼 엔지니어의 언어로 번역해야 대화가 시작됩니다.
- 덧붙여, 그 원인 지목이 실제로 맞는지 정답이 있는 구간에서 미리 채점해 두어야 신뢰를 얻을 수 있습니다.

</details>


---
## 다음 노트북 예고 — NB05. 표현학습(SSL)과 도메인 적응

난제 ①은 회수했습니다. 그런데 오늘 실습에는 조용한 전제가 하나 깔려 있었습니다.
**"학습에 쓸 정상 데이터가 충분히 있다"** 는 것입니다.

- 지난주 증설한 신규 라인에는 그 정상 데이터조차 없습니다 → **난제 ② Cold-Start**
- 어제까지 맞던 모델이 부품 교체 한 번에 무너집니다 → **난제 ③ Concept Drift**

다음 노트북에서는 **라벨 없이 표현을 배우는 자기지도학습(SSL)** 으로 ②를,
**도메인 적응(MMD)** 으로 ③을 회수합니다.
NB03에서 다룬 엔진 데이터의 FD001 → FD003 전이가 무대이고,
*같은 엔진인데 운전 조건이 달라 모델이 무너지는* 장면을 실데이터로 확인합니다.